## Imports

In [1]:
import os
import re

import numpy as np
import pandas as pd
import hiplot as hip
import matplotlib.pyplot as plt
import datetime as datetime

from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore')

In [2]:
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 300)

In [3]:
## Connection
connection = presto.connect(
        host='presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

## Datasets & Parameter

In [4]:
# Get the current working directory
cwd = os.getcwd()
print(cwd)
local_datasets = '/Users/rapido/local-datasets/customer/appography/user_base_analysis/raw/'
print(local_datasets)

/Users/rapido/analytics/customer/Appography/user_base_analysis
/Users/rapido/local-datasets/customer/appography/user_base_analysis/raw/


### customer_installed_apps

Scaling started from 20240219

##customer_installed_apps

customer_installed_apps_query = f"""

    SELECT
        *
    FROM
        (
        SELECT
            id user, 
            yyyymmdd,
            json_extract(data, '$.appographyData') as app_list,
            row_number() over(partition by id order by updated_epoch desc) updated_seq
        FROM 
            raw_internal.kafka_info_customer_installed_apps_v1
        WHERE 
            yyyymmdd >= DATE_FORMAT((DATE_TRUNC('month', CURRENT_DATE) - INTERVAL '2' MONTH), '%Y%m%d')
            AND yyyymmdd <= DATE_FORMAT((CURRENT_DATE - INTERVAL '1' DAY), '%Y%m%d')
        )
    WHERE
        updated_seq = 1
"""

df_customer_installed_apps = pd.read_sql(customer_installed_apps_query, connection)
df_customer_installed_apps.to_csv(local_datasets + 'raw_customer_installed_apps.csv', index=False)

In [5]:
df_customer_installed_apps_raw = pd.read_csv(local_datasets + 'raw_customer_installed_apps.csv')
df_customer_installed_apps = df_customer_installed_apps_raw[['user', 'app_list']].copy()

## Define a function to extract values between double quotations and trim extra spaces
def extract_and_trim(text):
    return [val.strip() for val in re.findall(r'"(.*?)"', text)]

df_customer_installed_apps['app_list'] = df_customer_installed_apps['app_list'].apply(lambda x: extract_and_trim(x))

print(df_customer_installed_apps.shape)
# df_customer_installed_apps.head(3)

(5122100, 2)


### category_list

https://docs.google.com/spreadsheets/d/1O6IsJ2dzzhXKuoHFoOA0AD4s4SOerMC1U-KrpeFnbwQ/edit#gid=1742529009

category_list = pd.read_clipboard()
category_list.to_csv(local_datasets + 'category_list_v1.csv', index=False)

In [6]:
category_list = pd.read_csv(local_datasets + 'category_list_v1.csv')
category_list = category_list[['app_name', 'categories_l2', 'categories_l1']]

In [7]:
# category_list.head(2)

In [8]:
category_list.categories_l1.unique()

array(['Educational', 'OTT', 'Delivery_Grocery', 'Tools', 'Office',
       'Dating', 'News', 'Devotional', 'Travel_Metro', 'Gaming',
       'Travel_Bookings', 'Finance_Transactions', 'Ecommerce',
       'Streaming_Music', 'Finance_Investment', 'Health',
       'Fantasy_Sports', 'Delivery_Food', 'Home_Security', 'Finance_News',
       'Travel_Ridehailing', 'Social', 'Telecom', 'Entertainment',
       'News_finance', 'Wearable', 'Driver_App'], dtype=object)

In [9]:
category_list.categories_l2.unique()

array(['Educational', 'Rest', 'Office', 'Ride haling', 'Gaming',
       'Finance_Investment', 'Driver_App'], dtype=object)

### Active customer (RR-customers)

In [10]:
## Parameter 
start_date = '20240219'
end_date = '20240301'

##active_customer_query

active_customer_demographic_query = f"""

WITH active_customers AS (

SELECT 
    channel_host,
    customer_id,
    COALESCE(city_name, 'NA') city_name
FROM 
    (
    SELECT 
        channel_host,
        customer_id,
        city_name,
        row_number() over(partition by customer_id order by updated_epoch DESC) updated_seq
    FROM 
        orders.order_logs_snapshot
    WHERE
        yyyymmdd >= '{start_date}'
        AND yyyymmdd <= '{end_date}'
        AND channel_host IN ('android', 'ios')
    ) 
WHERE
    updated_seq = 1
), 

customer_single_view AS (

SELECT
    customerId,
    gender,
    lifetimeage rapido_age,
    CASE 
    WHEN ((POSITION('/' IN dateofbirth) > 0 AND dateofbirth <> 'NaN/NaN/NaN') or POSITION('-' IN dateofbirth) > 0) 
    THEN 
        CASE 
        WHEN TRY(date_diff('year', CAST(dateofbirth AS DATE), current_date)) IS NOT NULL 
        THEN date_diff('year', CAST(dateofbirth AS DATE), current_date)
        ELSE NULL
        END
    ELSE NULL
    END AS customer_age,
    lifetimerides
FROM 
    datasets.customer_single_view
WHERE 
    customerId IN (SELECT customer_id FROM active_customers)
),

iallocator_customer_segments AS (

SELECT
    customer_id user_id,
    taxi_ltr_segment ltr_segment,
    taxi_lifetime_rides life_time_rides,
    taxi_lifetime_stage life_time_stage,
    taxi_income_segment income_segment,
    taxi_service_affinity service_affinity,
    taxi_distance_preference distance_preference,
    CASE 
    WHEN ps_tag_auto = ps_tag_link THEN ps_tag_auto
    WHEN ps_tag_auto = 'PS' OR ps_tag_link = 'PS' THEN 'PS'
    WHEN ps_tag_auto = 'NPS' OR ps_tag_link = 'NPS' THEN 'NPS'
    ELSE 'UNKNOWN'
    END ps_tag_taxi
    
FROM 
    datasets.iallocator_customer_segments
WHERE 
    run_date = '2024-03-01'
    AND customer_id IN (SELECT customer_id FROM active_customers)
)

SELECT
    a.*,
    c.gender,
    c.rapido_age,
    COALESCE(CAST(c.customer_age AS VARCHAR), 'NA') customer_age,
    CASE 
    WHEN life_time_rides = 0 OR c.lifetimerides = 0 THEN 'LTR 0'
    WHEN life_time_rides > 0 THEN ltr_segment
    ELSE 'UNKNOWN'
    END AS ltr_segment,
    COALESCE(life_time_rides, c.lifetimerides) life_time_rides,
    COALESCE(life_time_stage, 'UNKNOWN') AS life_time_stage,
    COALESCE(income_segment, 'UNKNOWN') AS income_segment,
    COALESCE(service_affinity, 'UNKNOWN') AS service_affinity,
    COALESCE(distance_preference, 'UNKNOWN') AS distance_preference,
    COALESCE(ps_tag_taxi, 'UNKNOWN') AS price_sensitivity 
FROM 
    active_customers a
LEFT JOIN 
    customer_single_view c
    ON a.customer_id = c.customerId
LEFT JOIN 
    iallocator_customer_segments b
    ON a.customer_id = b.user_id
"""

df_active_customer_demographic = pd.read_sql(active_customer_demographic_query, connection)
df_active_customer_demographic.to_csv(local_datasets + 'df_active_customer_demographic.csv', index=False)


active_customer_ao_net_query = f"""

WITH fe_tbl AS (
    SELECT
        DATE_FORMAT(DATE_PARSE(yyyymmdd, '%Y%m%d'), '%Y-%m-%d') order_date,
        city,
        service_name,
        user_id as user_id,
        fare_estimate_id
    FROM
        hive.pricing.fare_estimates_enriched
    WHERE 
        yyyymmdd >= '{start_date}'
        AND yyyymmdd <= '{end_date}'
),

rr_net_tbl as (
    SELECT   
        DATE_FORMAT(DATE_PARSE(yyyymmdd, '%Y%m%d'), '%Y-%m-%d') order_date,
        city_name AS city,
        service_obj_service_name AS service_name,
        customer_id,
        estimate_id AS fare_estimate_id,
        order_id,
        order_status,
        spd_fraud_flag
    FROM orders.order_logs_snapshot ols
    WHERE
        yyyymmdd >= '{start_date}'
        AND yyyymmdd <= '{end_date}'
        AND channel_host IN ('android', 'ios')
),

fe_merged AS (
    SELECT
        fe_tbl.*,
        rr_net_tbl.customer_id,
        rr_net_tbl.order_id,
        rr_net_tbl.order_status,
        rr_net_tbl.spd_fraud_flag
    FROM
        fe_tbl
    LEFT JOIN
        rr_net_tbl
        ON fe_tbl.order_date = rr_net_tbl.order_date
        AND fe_tbl.user_id = rr_net_tbl.customer_id
)

SELECT
    customer_id,
    COUNT(distinct fare_estimate_id) AS fe_count,
    COUNT(distinct order_id) AS rr_count,
    COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != True THEN order_id END) AS net_count
    
FROM 
    fe_merged
GROUP BY 1

"""

df_active_customer_ao_net_query = pd.read_sql(active_customer_ao_net_query, connection)
df_active_customer_ao_net_query.to_csv(local_datasets + 'df_active_customer_ao_net.csv', index=False)

In [96]:
df_active_customer_demographic = pd.read_csv(local_datasets + 'df_active_customer_demographic.csv')
df_active_customer_ao_net_query = pd.read_csv(local_datasets + 'df_active_customer_ao_net.csv')

## active_customer_demographic & funnel
df_active_customer = pd.merge(df_active_customer_demographic,
                              df_active_customer_ao_net_query,
                              how='left',
                              on = ['customer_id']
                             )
df_active_customer['fe2rr']=(df_active_customer['rr_count']*100.00/df_active_customer['fe_count']).round(2)
df_active_customer['g2n']=(df_active_customer['net_count']*100.00/df_active_customer['rr_count']).round(2)
df_active_customer['fe2net']=(df_active_customer['net_count']*100.00/df_active_customer['fe_count']).round(2)

df_active_customer.to_csv(local_datasets + 'df_active_customer.csv', index=False)
df_active_customer = pd.read_csv(local_datasets + 'df_active_customer.csv')

## Data workaround

In [97]:
print(df_active_customer.shape)
df_active_customer.head(2)

df_active_customer['gender'] = df_active_customer['gender'].fillna('UNKNOWN')
print(df_active_customer.gender.unique())

(7929248, 19)
['MALE' 'FEMALE' 'OTHERS' 'UNKNOWN']


In [98]:
print(df_customer_installed_apps.shape)
df_customer_installed_apps.head(2)

(5122100, 2)


,user,app_list
0,5dcfa6c9010c5a4601079831,"[Aha, Adobe Scan, Amazon Prime Video, Clock, Z..."
1,6141bcf37f5ff71791063319,"[Facebook, Messenger, Google Maps, Google Phot..."


In [99]:
print(category_list.shape)
category_list.head(2)

(399, 3)


,app_name,categories_l2,categories_l1
0,IGNOU e-Content,Educational,Educational
1,Aha,Rest,OTT


### What is the distribution of mobile os among our customer base?

In [100]:
df_temp = df_active_customer.groupby(['channel_host']).agg({'customer_id':'nunique'}).reset_index()
df_temp['% distribuion'] = (df_temp['customer_id']*100/df_temp['customer_id'].sum()).round(2)
df_temp

,channel_host,customer_id,% distribuion
0,android,6670216,84.12
1,ios,1259032,15.88


### Which city should I choose?

In [101]:
df_active_customer_android = df_active_customer[df_active_customer['channel_host'].isin(['android'])]

df_customer_coverage = pd.merge(df_active_customer_android,
                                df_customer_installed_apps,
                                how='left',
                                left_on=['customer_id'], 
                                right_on=['user']
                                )

df_temp = df_customer_coverage\
            .groupby(['city_name'])\
            .agg(active_customers = pd.NamedAgg('customer_id','nunique'),
                 app_info_customers = pd.NamedAgg('user','nunique')
                )\
            .sort_values(['active_customers'],ascending=False)\
            .reset_index()
df_temp['coverage %'] = (df_temp['app_info_customers']*100/df_temp['active_customers']).round(2)
df_temp.head(10)

,city_name,active_customers,app_info_customers,coverage %
0,Hyderabad,1591937,757176,47.56
1,Bangalore,1052507,499817,47.49
2,Delhi,903068,479617,53.11
3,Chennai,714924,326359,45.65
4,Kolkata,222314,108793,48.94
5,Jaipur,218198,106731,48.91
6,Pune,139958,69723,49.82
7,Ahmedabad,134672,67904,50.42
8,Indore,129543,61819,47.72
9,Lucknow,113598,58727,51.70


### Proceeding with Hyderabad city

In [102]:
## Merge Active customers & customers apps details

df_customer_hyd_all = df_customer_coverage[(df_customer_coverage['city_name'].isin(['Hyderabad'])) 
                                           &
                                           (df_customer_coverage['user'].notna())]

df_customer_hyd = df_customer_hyd_all.copy()
"""
df_customer_hyd = df_customer_hyd_all[['customer_id', 'apps_list', 
                                       'ltr', 'ltr_segment'
#                                        'recency_segment', 'income_segment',
#                                        'frequency_segment', 'retention_segment',
#                                        'service_affinity', 'customer_service_segment',
#                                        'ps_tag_auto', 'ps_tag_link'
                                      ]]
df_customer_hyd.head(2)
"""

df_customer_hyd.head(2)

,channel_host,customer_id,city_name,gender,rapido_age,customer_age,ltr_segment,life_time_rides,life_time_stage,income_segment,service_affinity,distance_preference,price_sensitivity,fe_count,rr_count,net_count,fe2rr,g2n,fe2net,user,app_list
22,android,61b2c6c05499d32332a235af,Hyderabad,MALE,814.0,40.0,PHH,46.0,HOOK,UNKNOWN,ONLY_LINK,MEDIUM_DISTANCE,UNKNOWN,9.0,3.0,3.0,33.33,100.00,33.33,61b2c6c05499d32332a235af,"[Clock, Facebook, Flipkart, Google Sheets, Goo..."
32,android,65ddc607a5277d6ce8ed5f9f,Hyderabad,MALE,5.0,NaN,PHH,11.0,COMMITTED,MEDIUM_INCOME,ONLY_AUTO,MEDIUM_DISTANCE,UNKNOWN,19.0,17.0,13.0,89.47,76.47,68.42,65ddc607a5277d6ce8ed5f9f,"[Adobe Scan, Amazon Prime Video, Amazon Music,..."


### Customer level apps list and category mapping 

In [103]:
## Explode list and map all the categories

df_customer_hyd_explode = df_customer_hyd.explode('app_list')
df_customer_app_cat_mapping_v0 = pd.merge(df_customer_hyd_explode, category_list, 
                                       how='left', 
                                       left_on='app_list', right_on='app_name')

df_customer_mapped_v0 = df_customer_app_cat_mapping_v0\
                        .groupby(['customer_id', 'life_time_rides', 'ltr_segment', 'income_segment'])\
                        .agg(app_list = ('app_list', lambda x: list(set(x))),
                             app_count = ('app_list', 'nunique'),
                             categories_l1 = ('categories_l1', lambda x: list(set(x))),
                             categories_l1_count = ('categories_l1', 'nunique'),
                             categories_l2 = ('categories_l2', lambda x: list(set(x))),
                             categories_l2_count = ('categories_l2', 'nunique')
                            )\
                        .reset_index()

print(df_customer_mapped_v0.shape)
df_customer_mapped_v0.head(2)

(757176, 10)


,customer_id,life_time_rides,ltr_segment,income_segment,app_list,app_count,categories_l1,categories_l1_count,categories_l2,categories_l2_count
0,5737c6deddbec2203f733314,1.0,HH,MEDIUM_INCOME,"[Snapchat, Gmail, Clock, Meesho, Google Calend...",11,"[Social, Ecommerce, Tools]",3,[Rest],1
1,573f28f39b0ffc283676f290,102.0,PHH,HIGH_INCOME,"[Angel Broking, Ajio, Google Calendar, Zomato,...",49,"[OTT, Travel_Ridehailing, Finance_Transactions...",14,"[Ride haling, Finance_Investment, Office, Rest]",4


In [166]:
df_customer_mapped_v0['app_count'].describe()

count    757176.000000
mean         22.391848
std           9.085945
min           1.000000
25%          16.000000
50%          21.000000
75%          28.000000
max         101.000000
Name: app_count, dtype: float64

In [169]:
df_customer_mapped_v0['bucket_app_count'] = np.where(df_customer_mapped_v0['app_count'] < 17 , '1-16',
                                                     np.where(df_customer_mapped_v0['app_count'] < 30 , '17-30',
                                                              np.where(df_customer_mapped_v0['app_count'] < 50 , '31-50', 'above 50')))


In [172]:
test_df = df_customer_mapped_v0.customer_id.nunique()
df_temp = df_customer_mapped_v0.groupby(['bucket_app_count']).agg({'customer_id' : 'nunique'}).reset_index()

df_temp['%'] = (df_temp['customer_id']*100.00/test_df).round(2)
df_temp

,bucket_app_count,customer_id,%
0,1-16,219228,28.95
1,17-30,382203,50.48
2,31-50,150332,19.85
3,above 50,5413,0.71


In [173]:
#df_customer_mapped_v0.groupby(['app_count']).agg({'customer_id' : 'nunique'})

#### Distribution check on apps

In [104]:
total_customers = df_customer_hyd.customer_id.nunique()
total_customers

757176

In [174]:
df_customer_hyd_explode.app_list.nunique()

220

In [105]:
df_temp = df_customer_hyd_explode\
            .groupby(['app_list'])\
            .agg(customers = ('customer_id','nunique'))\
            .sort_values(['customers'],ascending=False)\
            .reset_index()
df_temp['%'] =  (df_temp['customers']*100.00/total_customers).round(2)
df_temp.head(50)

,app_list,customers,%
0,YouTube,756840,99.96
1,Google Maps,756777,99.95
2,Gmail,756570,99.92
3,WhatsApp,736494,97.27
4,Google Photos,720795,95.20
5,Instagram,593082,78.33
6,Facebook,510381,67.41
7,Truecaller,500228,66.06
8,Paytm Money,479440,63.32
9,Google Calendar,446272,58.94


In [106]:
commonly_used_app_list = ['YouTube', 'Google Maps', 'Gmail', 'WhatsApp', 
                          'Google Photos','Google Calendar']

#### Distribution check on categories

In [107]:
df_temp = df_customer_app_cat_mapping_v0\
            .groupby(['categories_l1'])\
            .agg(customers = ('customer_id','nunique'))\
            .sort_values(['customers'],ascending=False)\
            .reset_index()
df_temp['%'] =  (df_temp['customers']*100.00/total_customers).round(2)
df_temp

,categories_l1,customers,%
0,Tools,757148,100.00
1,Social,755915,99.83
2,OTT,586376,77.44
3,Ecommerce,564540,74.56
4,Finance_Transactions,513409,67.81
5,Travel_Ridehailing,479123,63.28
6,Streaming_Music,467070,61.69
7,Delivery_Food,464892,61.40
8,Office,392195,51.80
9,News,254938,33.67


In [108]:
df_temp = df_customer_app_cat_mapping_v0\
            .groupby(['categories_l2'])\
            .agg(customers = ('customer_id','nunique'))\
            .sort_values(['customers'],ascending=False)\
            .reset_index()
df_temp['%'] =  (df_temp['customers']*100.00/total_customers).round(2)
df_temp

,categories_l2,customers,%
0,Rest,757175,100.00
1,Ride haling,479123,63.28
2,Office,392195,51.80
3,Finance_Investment,186909,24.69
4,Educational,125357,16.56
5,Gaming,122845,16.22
6,Driver_App,76593,10.12


#### Consider removing commonly used apps since it's almost present in all the customers mobile

In [109]:
commonly_used_app_list

['YouTube',
 'Google Maps',
 'Gmail',
 'WhatsApp',
 'Google Photos',
 'Google Calendar']

In [110]:
## Explode list and map all the categories

df_customer_hyd_explode = df_customer_hyd.explode('app_list')
df_customer_app_cat_mapping_v1 = pd.merge(df_customer_hyd_explode[~df_customer_hyd_explode['app_list'].isin(commonly_used_app_list)], 
                                          category_list, 
                                          how='left', 
                                          left_on='app_list', 
                                          right_on='app_name')

df_customer_mapped_v1 = df_customer_app_cat_mapping_v1\
                        .groupby(['customer_id', 'life_time_rides', 'ltr_segment','income_segment'])\
                        .agg(app_list = ('app_list', lambda x: list(set(x))),
                             app_count = ('app_list', 'nunique'),
                             categories_l1 = ('categories_l1', lambda x: list(set(x))),
                             categories_l1_count = ('categories_l1', 'nunique'),
                             categories_l2 = ('categories_l2', lambda x: list(set(x))),
                             categories_l2_count = ('categories_l2', 'nunique')
                            )\
                        .reset_index()

print(df_customer_mapped_v1.shape)
df_customer_mapped_v1.head(2)

(756428, 10)


,customer_id,life_time_rides,ltr_segment,income_segment,app_list,app_count,categories_l1,categories_l1_count,categories_l2,categories_l2_count
0,5737c6deddbec2203f733314,1.0,HH,MEDIUM_INCOME,"[Snapchat, Clock, Meesho, Instagram, Google Sl...",5,"[Social, Ecommerce, Tools]",3,[Rest],1
1,573f28f39b0ffc283676f290,102.0,PHH,HIGH_INCOME,"[Angel Broking, Ajio, Zomato, CRED, Truecaller...",43,"[OTT, Travel_Ridehailing, Finance_Transactions...",14,"[Ride haling, Finance_Investment, Office, Rest]",4


##### Sanity checks 

In [111]:
df_customer_mapped_v0[df_customer_mapped_v0['customer_id'] == '573f28f59b0ffc283676fd0f']

,customer_id,life_time_rides,ltr_segment,income_segment,app_list,app_count,categories_l1,categories_l1_count,categories_l2,categories_l2_count
9,573f28f59b0ffc283676fd0f,545.0,PHH,MEDIUM_INCOME,"[Ajio, Google Calendar, CRED, Zomato, Truecall...",27,"[OTT, Travel_Ridehailing, Streaming_Music, Fin...",9,"[Ride haling, Rest]",2


In [112]:
df_customer_mapped_v1[df_customer_mapped_v1['customer_id'] == '573f28f59b0ffc283676fd0f']

,customer_id,life_time_rides,ltr_segment,income_segment,app_list,app_count,categories_l1,categories_l1_count,categories_l2,categories_l2_count
9,573f28f59b0ffc283676fd0f,545.0,PHH,MEDIUM_INCOME,"[Ajio, CRED, Zomato, Truecaller, Cred, Axis Mo...",21,"[OTT, Travel_Ridehailing, Streaming_Music, Fin...",9,"[Ride haling, Rest]",2


#### Distribution check on categories (After removing 5 common apps)

In [113]:
df_temp = df_customer_app_cat_mapping_v1\
            .groupby(['categories_l1'])\
            .agg(app_list = ('app_list', lambda x: list(set(x))),
                 app_count = ('app_list', 'nunique'),
                 customers = ('customer_id','nunique'))\
            .sort_values(['customers'],ascending=False)\
            .reset_index()
df_temp['%'] =  (df_temp['customers']*100.00/total_customers).round(2)
df_temp

,categories_l1,app_list,app_count,customers,%
0,Social,"[Pinterest, Snapchat, Facebook, LinkedIn, Skyp...",17,732002,96.68
1,Tools,"[Hangouts, Microsoft OneNote, File Commander, ...",18,650223,85.87
2,OTT,"[Netflix, Amazon Prime Video, Aha, ZEE5, Disne...",9,586376,77.44
3,Ecommerce,"[Zivame, Club Factory, Ajio, FirstCry, Voonik,...",14,564540,74.56
4,Finance_Transactions,"[MobiKwik, FreeCharge, CRED, Cred, Paytm Money...",6,513409,67.81
5,Travel_Ridehailing,"[In Drive, Quick Ride, Blusmart, DriveU Driver...",9,479123,63.28
6,Streaming_Music,"[Wynk Music, Hungama Music, Gaana, SoundCloud,...",6,467070,61.69
7,Delivery_Food,"[Uber Eats, Dunzo, Zomato, Swiggy]",4,464892,61.40
8,Office,"[Github, Confluence, Zoho Meeting, MS Authenti...",23,392195,51.80
9,News,"[Google News, Feedly, The Indian Express]",3,254938,33.67


In [114]:
df_temp = df_customer_app_cat_mapping_v1\
            .groupby(['categories_l2'])\
            .agg(app_list = ('app_list', lambda x: list(set(x))),
                 app_count = ('app_list', 'nunique'),
                 customers = ('customer_id','nunique'))\
            .sort_values(['customers'],ascending=False)\
            .reset_index()
df_temp['%'] =  (df_temp['customers']*100.00/total_customers).round(2)
df_temp

,categories_l2,app_list,app_count,customers,%
0,Rest,"[MLB, FirstCry, Adobe Lightroom, FreeCharge, M...",124,755917,99.83
1,Ride haling,"[In Drive, Quick Ride, Blusmart, DriveU Driver...",9,479123,63.28
2,Office,"[Github, Confluence, Zoho Meeting, MS Authenti...",23,392195,51.80
3,Finance_Investment,"[HDFC Mobile Banking, Angel Broking, Namma Par...",6,186909,24.69
4,Educational,"[myCBSEguide, Goodreads, IGNOU e-Content, Mood...",41,125357,16.56
5,Gaming,"[Free Fire, Ludo King]",2,122845,16.22
6,Driver_App,"[Zepto driver app, Ola Driver app, Uber Driver...",10,76593,10.12


#### Analysis

In [115]:
### Ride haling

df_temp = df_customer_app_cat_mapping_v1[df_customer_app_cat_mapping_v1['categories_l2'].isin(['Ride haling'])]\
            .groupby(['categories_l2','app_list'])\
            .agg(customers = ('customer_id','nunique'))\
            .sort_values(['customers'],ascending=False)\
            .reset_index()
df_temp['%'] =  (df_temp['customers']*100.00/total_customers).round(2)
df_temp

,categories_l2,app_list,customers,%
0,Ride haling,Ola,369177,48.76
1,Ride haling,Uber,345450,45.62
2,Ride haling,Namma Yatri,9336,1.23
3,Ride haling,In Drive,9265,1.22
4,Ride haling,Quick Ride,2956,0.39
5,Ride haling,DriveU Driver app,2020,0.27
6,Ride haling,Blusmart,1249,0.16
7,Ride haling,Quickride,460,0.06
8,Ride haling,Jugnoo,279,0.04


In [116]:
### Office

df_temp = df_customer_app_cat_mapping_v1[df_customer_app_cat_mapping_v1['categories_l2'].isin(['Office'])]\
            .groupby(['categories_l2','app_list'])\
            .agg(customers = ('customer_id','nunique'))\
            .sort_values(['customers'],ascending=False)\
            .reset_index()
df_temp['%'] =  (df_temp['customers']*100.00/total_customers).round(2)
df_temp

,categories_l2,app_list,customers,%
0,Office,Zoom,197672,26.11
1,Office,OneDrive,173521,22.92
2,Office,Outlook,151290,19.98
3,Office,Microsoft 365,142647,18.84
4,Office,Microsoft Teams,119109,15.73
5,Office,MS Authenticator,76510,10.10
6,Office,Intune Company Portal,58284,7.70
7,Office,Google Authenticator,27389,3.62
8,Office,Webex,23848,3.15
9,Office,Slack,11083,1.46


In [117]:
### Finance_Investment

df_temp = df_customer_app_cat_mapping_v1[df_customer_app_cat_mapping_v1['categories_l2'].isin(['Finance_Investment'])]\
            .groupby(['categories_l2','app_list'])\
            .agg(customers = ('customer_id','nunique'))\
            .sort_values(['customers'],ascending=False)\
            .reset_index()
df_temp['%'] =  (df_temp['customers']*100.00/total_customers).round(2)
df_temp

,categories_l2,app_list,customers,%
0,Finance_Investment,HDFC Mobile Banking,117085,15.46
1,Finance_Investment,Groww,50288,6.64
2,Finance_Investment,Kite by Zerodha,36382,4.80
3,Finance_Investment,Angel Broking,23385,3.09
4,Finance_Investment,NPS,9193,1.21
5,Finance_Investment,Namma Partner,1276,0.17


In [118]:
### Educational

df_temp = df_customer_app_cat_mapping_v1[df_customer_app_cat_mapping_v1['categories_l2'].isin(['Educational'])]\
            .groupby(['categories_l2','app_list'])\
            .agg(customers = ('customer_id','nunique'))\
            .sort_values(['customers'],ascending=False)\
            .reset_index()
df_temp['%'] =  (df_temp['customers']*100.00/total_customers).round(2)
df_temp

,categories_l2,app_list,customers,%
0,Educational,Google Classroom,26293,3.47
1,Educational,Udemy,26016,3.44
2,Educational,Duolingo,23580,3.11
3,Educational,Brainly,20534,2.71
4,Educational,Byju,19555,2.58
5,Educational,Unacademy,11430,1.51
6,Educational,Coursera,6947,0.92
7,Educational,SWAYAM,5749,0.76
8,Educational,Physics Wallah,4976,0.66
9,Educational,Doubtnut,3793,0.50


In [119]:
### Gaming

df_temp = df_customer_app_cat_mapping_v1[df_customer_app_cat_mapping_v1['categories_l2'].isin(['Gaming'])]\
            .groupby(['categories_l2','app_list'])\
            .agg(customers = ('customer_id','nunique'))\
            .sort_values(['customers'],ascending=False)\
            .reset_index()
df_temp['%'] =  (df_temp['customers']*100.00/total_customers).round(2)
df_temp

,categories_l2,app_list,customers,%
0,Gaming,Ludo King,122844,16.22
1,Gaming,Free Fire,1,0.00


In [120]:
### Driver_App

df_temp = df_customer_app_cat_mapping_v1[df_customer_app_cat_mapping_v1['categories_l2'].isin(['Driver_App'])]\
            .groupby(['categories_l2','app_list'])\
            .agg(customers = ('customer_id','nunique'))\
            .sort_values(['customers'],ascending=False)\
            .reset_index()
df_temp['%'] =  (df_temp['customers']*100.00/total_customers).round(2)
df_temp

,categories_l2,app_list,customers,%
0,Driver_App,Rapido Captain,52749,6.97
1,Driver_App,Uber Driver app,26763,3.53
2,Driver_App,Ola Driver app,19467,2.57
3,Driver_App,Swiggy Driver app,12444,1.64
4,Driver_App,Porter Driver app,6685,0.88
5,Driver_App,Zomato Driver app,4958,0.65
6,Driver_App,Zepto driver app,3856,0.51
7,Driver_App,Shadowfax driver app,2867,0.38
8,Driver_App,Dunzo Driver app,877,0.12
9,Driver_App,Borzo Driver app,703,0.09


In [121]:
### Delivery_Food

df_temp = df_customer_app_cat_mapping_v1[df_customer_app_cat_mapping_v1['categories_l1'].isin(['Delivery_Food'])]\
            .groupby(['categories_l1','app_list'])\
            .agg(customers = ('customer_id','nunique'))\
            .sort_values(['customers'],ascending=False)\
            .reset_index()
df_temp['%'] =  (df_temp['customers']*100.00/total_customers).round(2)
df_temp

,categories_l1,app_list,customers,%
0,Delivery_Food,Zomato,381844,50.43
1,Delivery_Food,Swiggy,347545,45.90
2,Delivery_Food,Dunzo,23751,3.14
3,Delivery_Food,Uber Eats,2832,0.37


In [122]:
### Ecommerce

df_temp = df_customer_app_cat_mapping_v1[df_customer_app_cat_mapping_v1['categories_l1'].isin(['Ecommerce'])]\
            .groupby(['categories_l1','app_list'])\
            .agg(customers = ('customer_id','nunique'))\
            .sort_values(['customers'],ascending=False)\
            .reset_index()
df_temp['%'] =  (df_temp['customers']*100.00/total_customers).round(2)
df_temp

,categories_l1,app_list,customers,%
0,Ecommerce,Flipkart,440783,58.21
1,Ecommerce,Meesho,273245,36.09
2,Ecommerce,Myntra,245446,32.42
3,Ecommerce,Ajio,175816,23.22
4,Ecommerce,JioMart,62377,8.24
5,Ecommerce,Nykaa,48108,6.35
6,Ecommerce,FirstCry,41174,5.44
7,Ecommerce,PharmEasy,16324,2.16
8,Ecommerce,Zivame,9866,1.30
9,Ecommerce,Amazon Shopping,4254,0.56


In [123]:
### Streaming_Music

df_temp = df_customer_app_cat_mapping_v1[df_customer_app_cat_mapping_v1['categories_l1'].isin(['Streaming_Music'])]\
            .groupby(['categories_l1','app_list'])\
            .agg(customers = ('customer_id','nunique'))\
            .sort_values(['customers'],ascending=False)\
            .reset_index()
df_temp['%'] =  (df_temp['customers']*100.00/total_customers).round(2)
df_temp

,categories_l1,app_list,customers,%
0,Streaming_Music,Spotify,330706,43.68
1,Streaming_Music,Wynk Music,201111,26.56
2,Streaming_Music,Amazon Music,49936,6.60
3,Streaming_Music,Gaana,27758,3.67
4,Streaming_Music,SoundCloud,4364,0.58
5,Streaming_Music,Hungama Music,3690,0.49


### Work

In [124]:
df_customer_mapped_v1.head(5)

,customer_id,life_time_rides,ltr_segment,income_segment,app_list,app_count,categories_l1,categories_l1_count,categories_l2,categories_l2_count
0,5737c6deddbec2203f733314,1.0,HH,MEDIUM_INCOME,"[Snapchat, Clock, Meesho, Instagram, Google Sl...",5,"[Social, Ecommerce, Tools]",3,[Rest],1
1,573f28f39b0ffc283676f290,102.0,PHH,HIGH_INCOME,"[Angel Broking, Ajio, Zomato, CRED, Truecaller...",43,"[OTT, Travel_Ridehailing, Finance_Transactions...",14,"[Ride haling, Finance_Investment, Office, Rest]",4
2,573f28f39b0ffc283676f2a9,12.0,PHH,HIGH_INCOME,"[Fitbit, Zomato, Google Slides, Truecaller, Ki...",35,"[OTT, Travel_Ridehailing, Dating, Streaming_Mu...",17,"[Finance_Investment, Ride haling, Office, Rest]",4
3,573f28f49b0ffc283676f779,935.0,PHH,MEDIUM_INCOME,"[Netflix, Amazon Prime Video, Wynk Music, Face...",16,"[OTT, Travel_Ridehailing, Streaming_Music, Soc...",7,"[Ride haling, Rest]",2
4,573f28f59b0ffc283676fabf,42.0,PHH,UNKNOWN,"[MobiKwik, Adobe Lightroom, Zomato, CRED, Cred...",42,"[OTT, Travel_Ridehailing, Streaming_Music, Fin...",15,"[Ride haling, Rest, Educational, Finance_Inves...",5


In [125]:
data = df_customer_mapped_v1

## Explode all categories and perform a distribution analysis

In [126]:
df_temp = df_customer_hyd_explode\
            .groupby(['app_list'])\
            .agg(customers = ('customer_id','nunique'))\
            .sort_values(['customers'],ascending=False)\
            .reset_index()
df_temp['%'] =  (df_temp['customers']*100.00/total_customers).round(2)
df_temp.head(30)

,app_list,customers,%
0,YouTube,756840,99.96
1,Google Maps,756777,99.95
2,Gmail,756570,99.92
3,WhatsApp,736494,97.27
4,Google Photos,720795,95.20
5,Instagram,593082,78.33
6,Facebook,510381,67.41
7,Truecaller,500228,66.06
8,Paytm Money,479440,63.32
9,Google Calendar,446272,58.94


In [127]:
commonly_used_app_list

['YouTube',
 'Google Maps',
 'Gmail',
 'WhatsApp',
 'Google Photos',
 'Google Calendar']

In [128]:
## All apps
## df_customer_app_cat_mapping_v0.head(2)

## Removed 5 commonly used apps
## df_customer_app_cat_mapping_v1.head(2)

In [129]:
df_temp = df_customer_app_cat_mapping_v0\
            .groupby(['categories_l1'])\
            .agg(app_list = ('app_list', lambda x: list(set(x))),
                 app_count = ('app_list', 'nunique'),
                 customers = ('customer_id','nunique'))\
            .sort_values(['customers'],ascending=False)\
            .reset_index()
df_temp['%'] =  (df_temp['customers']*100.00/total_customers).round(2)
df_temp

,categories_l1,app_list,app_count,customers,%
0,Tools,"[Google Calendar, Adobe Lightroom, Notion, Goo...",23,757148,100.00
1,Social,"[Pinterest, Snapchat, Facebook, LinkedIn, Skyp...",18,755915,99.83
2,OTT,"[Netflix, Amazon Prime Video, Aha, ZEE5, Disne...",9,586376,77.44
3,Ecommerce,"[Zivame, Club Factory, Ajio, FirstCry, Voonik,...",14,564540,74.56
4,Finance_Transactions,"[MobiKwik, FreeCharge, CRED, Cred, Paytm Money...",6,513409,67.81
5,Travel_Ridehailing,"[In Drive, Quick Ride, Blusmart, DriveU Driver...",9,479123,63.28
6,Streaming_Music,"[Wynk Music, Hungama Music, Gaana, SoundCloud,...",6,467070,61.69
7,Delivery_Food,"[Uber Eats, Dunzo, Zomato, Swiggy]",4,464892,61.40
8,Office,"[Github, Confluence, Zoho Meeting, MS Authenti...",23,392195,51.80
9,News,"[Google News, Feedly, The Indian Express]",3,254938,33.67


In [130]:
df_temp = df_customer_app_cat_mapping_v1\
            .groupby(['categories_l1'])\
            .agg(app_list = ('app_list', lambda x: list(set(x))),
                 app_count = ('app_list', 'nunique'),
                 customers = ('customer_id','nunique'))\
            .sort_values(['customers'],ascending=False)\
            .reset_index()
df_temp['%'] =  (df_temp['customers']*100.00/total_customers).round(2)
df_temp

,categories_l1,app_list,app_count,customers,%
0,Social,"[Pinterest, Snapchat, Facebook, LinkedIn, Skyp...",17,732002,96.68
1,Tools,"[Hangouts, Microsoft OneNote, File Commander, ...",18,650223,85.87
2,OTT,"[Netflix, Amazon Prime Video, Aha, ZEE5, Disne...",9,586376,77.44
3,Ecommerce,"[Zivame, Club Factory, Ajio, FirstCry, Voonik,...",14,564540,74.56
4,Finance_Transactions,"[MobiKwik, FreeCharge, CRED, Cred, Paytm Money...",6,513409,67.81
5,Travel_Ridehailing,"[In Drive, Quick Ride, Blusmart, DriveU Driver...",9,479123,63.28
6,Streaming_Music,"[Wynk Music, Hungama Music, Gaana, SoundCloud,...",6,467070,61.69
7,Delivery_Food,"[Uber Eats, Dunzo, Zomato, Swiggy]",4,464892,61.40
8,Office,"[Github, Confluence, Zoho Meeting, MS Authenti...",23,392195,51.80
9,News,"[Google News, Feedly, The Indian Express]",3,254938,33.67


In [184]:
### Office

df_temp = df_customer_app_cat_mapping_v1[df_customer_app_cat_mapping_v1['categories_l1'].isin(['Travel_Ridehailing'])]\
            .groupby(['categories_l1','app_list'])\
            .agg(customers = ('customer_id','nunique'))\
            .sort_values(['customers'],ascending=False)\
            .reset_index()
df_temp['%'] =  (df_temp['customers']*100.00/total_customers).round(2)
office_total_customers = 479123
df_temp

,categories_l1,app_list,customers,%
0,Travel_Ridehailing,Ola,369177,48.76
1,Travel_Ridehailing,Uber,345450,45.62
2,Travel_Ridehailing,Namma Yatri,9336,1.23
3,Travel_Ridehailing,In Drive,9265,1.22
4,Travel_Ridehailing,Quick Ride,2956,0.39
5,Travel_Ridehailing,DriveU Driver app,2020,0.27
6,Travel_Ridehailing,Blusmart,1249,0.16
7,Travel_Ridehailing,Quickride,460,0.06
8,Travel_Ridehailing,Jugnoo,279,0.04


### Office

In [175]:
### Office

df_temp = df_customer_app_cat_mapping_v1[df_customer_app_cat_mapping_v1['categories_l1'].isin(['Office'])]\
            .groupby(['categories_l1','app_list'])\
            .agg(customers = ('customer_id','nunique'))\
            .sort_values(['customers'],ascending=False)\
            .reset_index()
df_temp['%'] =  (df_temp['customers']*100.00/total_customers).round(2)
office_total_customers = 392195
df_temp

,categories_l1,app_list,customers,%
0,Office,Zoom,197672,26.11
1,Office,OneDrive,173521,22.92
2,Office,Outlook,151290,19.98
3,Office,Microsoft 365,142647,18.84
4,Office,Microsoft Teams,119109,15.73
5,Office,MS Authenticator,76510,10.10
6,Office,Intune Company Portal,58284,7.70
7,Office,Google Authenticator,27389,3.62
8,Office,Webex,23848,3.15
9,Office,Slack,11083,1.46


In [131]:
df_customer_app_cat_mapping_v1.columns

Index(['channel_host', 'customer_id', 'city_name', 'gender', 'rapido_age',
       'customer_age', 'ltr_segment', 'life_time_rides', 'life_time_stage',
       'income_segment', 'service_affinity', 'distance_preference',
       'price_sensitivity', 'fe_count', 'rr_count', 'net_count', 'fe2rr',
       'g2n', 'fe2net', 'user', 'app_list', 'app_name', 'categories_l2',
       'categories_l1'],
      dtype='object')

In [154]:
### Office

df_temp = df_customer_app_cat_mapping_v1[df_customer_app_cat_mapping_v1['categories_l1'].isin(['Office'])]\
            .groupby(['categories_l1'])\
            .agg(customers = ('customer_id','nunique'))\
            .sort_values(['customers'],ascending=False)\
            .reset_index()
df_temp['%'] =  (df_temp['customers']*100.00/total_customers).round(2)
office_total_customers = 392195
df_temp

,categories_l1,customers,%
0,Office,392195,51.8


In [162]:
### Office


df_temp = df_customer_app_cat_mapping_v1[df_customer_app_cat_mapping_v1['categories_l1'].isin(['Office'])]\
            .groupby(['categories_l1','ltr_segment'])\
            .agg(customers = ('customer_id','nunique'))\
            .sort_values(['customers'],ascending=False)\
            .reset_index()
df_temp['%'] =  (df_temp['customers']*100.00/office_total_customers).round(2)
print(df_customer_app_cat_mapping_v1.gender.unique())
df_temp

['MALE' 'FEMALE' 'UNKNOWN' 'OTHERS']


,categories_l1,ltr_segment,customers,%
0,Office,PHH,316773,80.77
1,Office,HH,63663,16.23
2,Office,LTR 0,8868,2.26
3,Office,UNKNOWN,2891,0.74


In [177]:
### feedback - LTR buckets 
### Retention # on office

In [155]:
### Office

df_temp = df_customer_app_cat_mapping_v1[df_customer_app_cat_mapping_v1['categories_l1'].isin(['Office'])]\
            .groupby(['categories_l1','gender'])\
            .agg(customers = ('customer_id','nunique'))\
            .sort_values(['customers'],ascending=False)\
            .reset_index()
df_temp['%'] =  (df_temp['customers']*100.00/office_total_customers).round(2)
print(df_customer_app_cat_mapping_v1.gender.unique())
df_temp

['MALE' 'FEMALE' 'UNKNOWN' 'OTHERS']


,categories_l1,gender,customers,%
0,Office,MALE,250729,63.93
1,Office,FEMALE,129460,33.01
2,Office,UNKNOWN,10847,2.77
3,Office,OTHERS,1159,0.30


In [156]:

df_temp = df_customer_app_cat_mapping_v1[df_customer_app_cat_mapping_v1['categories_l1'].isin(['Office'])]\
            .groupby(['categories_l1','life_time_stage'])\
            .agg(customers = ('customer_id','nunique'))\
            .sort_values(['customers'],ascending=False)\
            .reset_index()
df_temp['%'] =  (df_temp['customers']*100.00/office_total_customers).round(2)
df_temp

,categories_l1,life_time_stage,customers,%
0,Office,HOOK,126407,32.23
1,Office,COMMITTED,100715,25.68
2,Office,HANDHOLDING,53199,13.56
3,Office,SUSTENANCE,43892,11.19
4,Office,DORMANT,22084,5.63
5,Office,INACTIVE,15839,4.04
6,Office,UNKNOWN,11624,2.96
7,Office,CHURN_OTB,11165,2.85
8,Office,SOFT_CHURN,5626,1.43
9,Office,DETOX,1644,0.42


In [146]:
df_customer_app_cat_mapping_v1.columns

Index(['channel_host', 'customer_id', 'city_name', 'gender', 'rapido_age',
       'customer_age', 'ltr_segment', 'life_time_rides', 'life_time_stage',
       'income_segment', 'service_affinity', 'distance_preference',
       'price_sensitivity', 'fe_count', 'rr_count', 'net_count', 'fe2rr',
       'g2n', 'fe2net', 'user', 'app_list', 'app_name', 'categories_l2',
       'categories_l1', 'age_bucket'],
      dtype='object')

In [157]:

df_temp = df_customer_app_cat_mapping_v1[df_customer_app_cat_mapping_v1['categories_l1'].isin(['Office'])]\
            .groupby(['categories_l1','income_segment'])\
            .agg(customers = ('customer_id','nunique'))\
            .sort_values(['customers'],ascending=False)\
            .reset_index()
df_temp['%'] =  (df_temp['customers']*100.00/office_total_customers).round(2)
df_temp

,categories_l1,income_segment,customers,%
0,Office,HIGH_INCOME,170501,43.47
1,Office,MEDIUM_INCOME,151120,38.53
2,Office,UNKNOWN,53585,13.66
3,Office,LOW_INCOME,16989,4.33


In [158]:

df_temp = df_customer_app_cat_mapping_v1[df_customer_app_cat_mapping_v1['categories_l1'].isin(['Office'])]\
            .groupby(['categories_l1','service_affinity'])\
            .agg(customers = ('customer_id','nunique'))\
            .sort_values(['customers'],ascending=False)\
            .reset_index()
df_temp['%'] =  (df_temp['customers']*100.00/office_total_customers).round(2)
df_temp

,categories_l1,service_affinity,customers,%
0,Office,ONLY_AUTO,155512,39.65
1,Office,ONLY_LINK,154793,39.47
2,Office,BOTH,54797,13.97
3,Office,UNKNOWN,27093,6.91


In [182]:
test_df = df_customer_app_cat_mapping_v0.customer_id.nunique()
df_temp = df_customer_app_cat_mapping_v0.groupby(['gender', 'service_affinity']).agg({'customer_id' : 'nunique'}).reset_index()

df_temp['%'] = (df_temp['customer_id']*100.00/test_df).round(2)
df_temp

,gender,service_affinity,customer_id,%
0,FEMALE,BOTH,25839,3.41
1,FEMALE,ONLY_AUTO,126202,16.67
2,FEMALE,ONLY_LINK,49763,6.57
3,FEMALE,UNKNOWN,12400,1.64
4,MALE,BOTH,76721,10.13
5,MALE,ONLY_AUTO,146748,19.38
6,MALE,ONLY_LINK,254160,33.57
7,MALE,UNKNOWN,41472,5.48
8,OTHERS,BOTH,389,0.05
9,OTHERS,ONLY_AUTO,1413,0.19


In [180]:
test_df

757176

In [159]:
df_temp = df_customer_app_cat_mapping_v1[df_customer_app_cat_mapping_v1['categories_l1'].isin(['Office'])]\
            .groupby(['categories_l1','distance_preference'])\
            .agg(customers = ('customer_id','nunique'))\
            .sort_values(['customers'],ascending=False)\
            .reset_index()
df_temp['%'] =  (df_temp['customers']*100.00/office_total_customers).round(2)
df_temp

,categories_l1,distance_preference,customers,%
0,Office,UNKNOWN,106964,27.27
1,Office,SHORT_DISTANCE,100044,25.51
2,Office,MEDIUM_DISTANCE,92928,23.69
3,Office,LONG_DISTANCE,92259,23.52


In [160]:
df_temp = df_customer_app_cat_mapping_v1[df_customer_app_cat_mapping_v1['categories_l1'].isin(['Office'])]\
            .groupby(['categories_l1','price_sensitivity'])\
            .agg(customers = ('customer_id','nunique'))\
            .sort_values(['customers'],ascending=False)\
            .reset_index()
df_temp['%'] =  (df_temp['customers']*100.00/office_total_customers).round(2)
df_temp

,categories_l1,price_sensitivity,customers,%
0,Office,UNKNOWN,182878,46.63
1,Office,NPS,114595,29.22
2,Office,PS,94722,24.15


In [161]:
### Office
df_customer_app_cat_mapping_v1['age_bucket'] = np.where(df_customer_app_cat_mapping_v1['customer_age'] < 18 , '< 18',
                                                          np.where(df_customer_app_cat_mapping_v1['customer_age'] < 23 , '18-22', 
                                                                   np.where(df_customer_app_cat_mapping_v1['customer_age'] < 35 , '23-35', 
                                                                            np.where(df_customer_app_cat_mapping_v1['customer_age'] < 65 , '45-65', '#NA'))))

df_temp = df_customer_app_cat_mapping_v1[df_customer_app_cat_mapping_v1['categories_l1'].isin(['Office'])]\
            .groupby(['categories_l1','age_bucket'])\
            .agg(customers = ('customer_id','nunique'))\
            .sort_values(['customers'],ascending=False)\
            .reset_index()
df_temp['%'] =  (df_temp['customers']*100.00/office_total_customers).round(2)
df_temp

,categories_l1,age_bucket,customers,%
0,Office,#NA,231038,58.91
1,Office,23-35,94169,24.01
2,Office,45-65,40047,10.21
3,Office,18-22,19721,5.03
4,Office,< 18,7220,1.84
